# How to see which funds bought or sold a stock from 13F filings

Use the change field on 13F holdings to list new positions, increases and full exits in one stock for a quarter, without joining two quarters yourself.

Read it with charts at https://www.quantisnow.com/sec-api/guides/which-funds-bought-a-stock. More notebooks: https://github.com/quantisnow/sec-edgar-notebooks. Set `QUANTISNOW_API_KEY` before running; make a key at https://www.quantisnow.com/sec-api/keys.

A 13F is a snapshot: what a manager held on the last day of the quarter. The question people actually ask, who bought and who sold, needs last quarter's snapshot next to this one. Every record in the 13F API already carries `previousShares` and a `change` of `new`, `added`, `trimmed`, `unchanged` or `sold`, so it is a filter.

## New positions in one stock

### Which funds opened a new position in Nvidia last quarter?

New positions are the clearest read of fresh conviction in a 13F, larger ones first.

```shell
curl -X POST https://www.quantisnow.com/api/v1/13f/holdings \
  -H "Authorization: Bearer $QUANTISNOW_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
  "query": "issuer.ticker:NVDA AND change:new AND period:2026-06-30",
  "from": 0,
  "size": 5,
  "sort": [{"valueUsd": {"order": "desc"}}]
}'
```

Response (real, first 2 of 216):

```json
{
  "total": {
    "value": 216,
    "relation": "eq"
  },
  "data": [
    {
      "id": "0001313360-26-000003:1045810",
      "period": "2026-06-30",
      "filedOn": "2026-08-07",
      "accessionNo": "0001313360-26-000003",
      "manager": {
        "cik": "0001313360",
        "name": "SG Americas Securities, LLC",
        "city": "NEW YORK",
        "state": "NY"
      },
      "issuer": {
        "cik": "0001045810",
        "ticker": "NVDA",
        "name": "NVIDIA Corporation"
      },
      "shares": 62298007,
      "valueUsd": 12465208000,
      "previousShares": 0,
      "change": "new",
      "changeValueUsd": 12465208000
    },
    {
      "id": "0001213900-26-088240:1045810",
      "period": "2026-06-30",
      "filedOn": "2026-08-12",
      "accessionNo": "0001213900-26-088240",
      "manager": {
        "cik": "0001633516",
        "name": "NewEdge Advisors, LLC",
        "city": "NEW ORLEANS",
        "state": "LA"
      },
      "issuer": {
        "cik": "0001045810",
        "ticker": "NVDA",
        "name": "NVIDIA Corporation"
      },
      "shares": 3483179,
      "valueUsd": 696949263,
      "previousShares": 0,
      "change": "new",
      "changeValueUsd": 696949263
    }
  ]
}
```

## Full exits

### Which funds sold all their Tesla shares last quarter?

Full exits, biggest first, to see who gave up on a name.

```shell
curl -X POST https://www.quantisnow.com/api/v1/13f/holdings \
  -H "Authorization: Bearer $QUANTISNOW_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
  "query": "issuer.ticker:TSLA AND change:sold AND period:2026-06-30",
  "from": 0,
  "size": 5,
  "sort": [{"changeValueUsd": {"order": "asc"}}]
}'
```

Response (real, first 2 of 146):

```json
{
  "total": {
    "value": 146,
    "relation": "eq"
  },
  "data": [
    {
      "id": "0001556921-26-000006:1318605",
      "period": "2026-06-30",
      "filedOn": "2026-08-14",
      "accessionNo": "0001556921-26-000006",
      "manager": {
        "cik": "0001556921",
        "name": "VOLORIDGE INVESTMENT MANAGEMENT, LLC",
        "city": "JUPITER",
        "state": "FL"
      },
      "issuer": {
        "cik": "0001318605",
        "ticker": "TSLA",
        "name": "Tesla Inc."
      },
      "shares": 0,
      "valueUsd": 0,
      "previousShares": 662844,
      "change": "sold",
      "changeValueUsd": -246412257
    },
    {
      "id": "0001651424-26-000010:1318605",
      "period": "2026-06-30",
      "filedOn": "2026-08-07",
      "accessionNo": "0001651424-26-000010",
      "manager": {
        "cik": "0001651424",
        "name": "Quadrature Capital Ltd",
        "city": "LONDON",
        "state": "X0"
      },
      "issuer": {
        "cik": "0001318605",
        "ticker": "TSLA",
        "name": "Tesla Inc."
      },
      "shares": 0,
      "valueUsd": 0,
      "previousShares": 489046,
      "change": "sold",
      "changeValueUsd": -181802851
    }
  ]
}
```

## Net flow for the quarter

In [ ]:
import os, requests

URL = "https://www.quantisnow.com/api/v1/13f/holdings"
KEY = os.environ["QUANTISNOW_API_KEY"]

def total(change):
    q = f"issuer.ticker:NVDA AND period:2026-06-30 AND change:{change}"
    page = requests.post(URL, headers={"Authorization": f"Bearer {KEY}"}, json={"query": q, "size": 1}).json()
    return page["total"]["value"]

print({c: total(c) for c in ["new", "added", "trimmed", "sold"]})

`size: 1` is enough when you only want the count: `total.value` is exact up to 10,000. For the dollar flow, page through `change:(new OR added OR trimmed OR sold)` and add up `changeValueUsd`.

13Fs are due 45 days after the quarter ends, so most of a quarter's filings arrive in the two weeks before that deadline. A count taken on day 30 is missing most funds.